# Q-Learning в Gymnasium: минимальная тетрадь Frozen Lake

Исполнимая версия урока `Frozen_Lake_Gymnasium.learning.md`: формулы, таблицы «формула → код» и разборы — в уроке.

Исходная тетрадь автора: `09.10.26_Frosen_Lake_minimal.ipynb`. Добавлены зерно 23, подписи действий в Q-таблице и карта политики.

```bash
pip install "numpy>=2.0" "gymnasium>=1.0"
```

## Шаг 1. Создаём среду

In [1]:
import gymnasium as gym
import numpy as np

SEED = 23
rng = np.random.default_rng(SEED)                        # (1)
np.set_printoptions(precision=4, suppress=True, floatmode="fixed")

env = gym.make("FrozenLake-v1", is_slippery=False)       # (2)
env.action_space.seed(SEED)                              # (3)
env.reset(seed=SEED)
n_states = env.observation_space.n
n_actions = env.action_space.n
cells = "".join(b"".join(row).decode() for row in env.unwrapped.desc)
ARROWS = "←↓→↑"                                          # (4)

alpha, gamma = 0.1, 0.99
epsilon_min, decay = 0.01, 0.9995
n_episodes, n_eval = 20000, 1000
print(f"states = {n_states}, actions = {n_actions}")
print("map =", "/".join(cells[i:i + 4] for i in range(0, 16, 4)))

states = 16, actions = 4
map = SFFF/FHFH/FFFH/HFFG


## Шаг 2. Делаем один эпизод вручную

In [2]:
s, _ = env.reset()
for a in (1, 1, 2, 1, 2, 2):                             # (1)
    s_next, r, term, trunc, _ = env.step(a)              # (2)
    print(f"s = {s:2d}  a = {a}{ARROWS[a]}  s' = {s_next:2d}  "
          f"r = {r}  term = {term}  trunc = {trunc}")
    s = s_next                                           # (3)

s =  0  a = 1↓  s' =  4  r = 0  term = False  trunc = False
s =  4  a = 1↓  s' =  8  r = 0  term = False  trunc = False
s =  8  a = 2→  s' =  9  r = 0  term = False  trunc = False
s =  9  a = 1↓  s' = 13  r = 0  term = False  trunc = False
s = 13  a = 2→  s' = 14  r = 0  term = False  trunc = False
s = 14  a = 2→  s' = 15  r = 1  term = True  trunc = False


## Шаг 3. Проверяем ручной расчёт

In [3]:
Q_demo = np.zeros((n_states, n_actions))
s, a, s_next, r, term = 14, 2, 15, 1, True              # (1)
for m in (1, 2):
    Q_demo[s, a] += alpha * (
        r + gamma * np.max(Q_demo[s_next]) * (not term) - Q_demo[s, a]
    )                                                    # (2)
    print(f"update {m}: Q[14, 2] = {Q_demo[s, a]:.4f}")
assert np.isclose(Q_demo[14, 2], 0.19)

update 1: Q[14, 2] = 0.1000
update 2: Q[14, 2] = 0.1900


## Шаг 4. Расписание исследования

In [4]:
epsilon = 1.0
history = []
for k in range(n_episodes):                              # (1)
    history.append(epsilon)
    epsilon = max(epsilon_min, epsilon * decay)          # (2)
history = np.array(history)
closed_form = np.maximum(epsilon_min, decay ** np.arange(n_episodes))
assert np.allclose(history, closed_form)                 # (3)
k_floor = int(np.argmax(history == epsilon_min))         # (4)
print(f"epsilon_0 = {history[0]:.4f}, epsilon_999 = {history[999]:.4f}")
print(f"first k with epsilon = epsilon_min: {k_floor}")

epsilon_0 = 1.0000, epsilon_999 = 0.6068
first k with epsilon = epsilon_min: 9209


## Шаг 5. Обучаем агента

In [5]:
CHECKPOINTS = {1, 100, 1000, 5000, 10000, 20000}
Q = np.zeros((n_states, n_actions))                      # (1)
epsilon, wins, trace = 1.0, 0, []
for k in range(n_episodes):                              # (2)
    s, _ = env.reset()
    done = False
    while not done:                                      # (3)
        if rng.random() < epsilon:                       # (4)
            a = int(env.action_space.sample())
        else:
            a = int(np.argmax(Q[s]))
        s_next, r, term, trunc, _ = env.step(a)
        Q[s, a] += alpha * (
            r + gamma * np.max(Q[s_next]) * (not term) - Q[s, a]
        )                                                # (5)
        s, done = s_next, term or trunc                  # (6)
    wins += r                                            # (7)
    if k + 1 in CHECKPOINTS:
        trace.append((k + 1, epsilon, wins, Q[14, 2], Q[0, 1]))
    epsilon = max(epsilon_min, epsilon * decay)

In [6]:
print("episode  epsilon   wins  Q[14,R]  Q[0,D]")
for k, eps_k, wins_k, q_goal, q_start in trace:          # (1)
    print(f"{k:7d}  {eps_k:7.4f}  {wins_k:5d}  "
          f"{q_goal:7.4f}  {q_start:6.4f}")

episode  epsilon   wins  Q[14,R]  Q[0,D]
      1   1.0000      0   0.0000  0.0000
    100   0.9517      2   0.1900  0.0003
   1000   0.6068    112   1.0000  0.9506
   5000   0.0821   2933   1.0000  0.9510
  10000   0.0100   7778   1.0000  0.9510
  20000   0.0100  17675   1.0000  0.9510


## Шаг 6. Печатаем таблицу с подписями действий

In [7]:
header = "".join(f"{a:>6d} {ARROWS[a]}" for a in range(n_actions))
print(f" s cell{header}")                                # (1)
for s in range(n_states):
    values = "".join(f"{q:8.4f}" for q in Q[s])          # (2)
    print(f"{s:2d}    {cells[s]}{values}")

 s cell     0 ←     1 ↓     2 →     3 ↑
 0    S  0.9415  0.9510  0.9321  0.9415
 1    F  0.9415  0.0000  0.9189  0.9312
 2    F  0.9312  0.7344  0.3604  0.6157
 3    F  0.6654  0.0000  0.1005  0.1097
 4    F  0.9510  0.9606  0.0000  0.9415
 5    H  0.0000  0.0000  0.0000  0.0000
 6    F  0.0000  0.9750  0.0000  0.3117
 7    H  0.0000  0.0000  0.0000  0.0000
 8    F  0.9606  0.0000  0.9703  0.9510
 9    F  0.9606  0.9801  0.9801  0.0000
10    F  0.9677  0.9900  0.0000  0.9330
11    H  0.0000  0.0000  0.0000  0.0000
12    H  0.0000  0.0000  0.0000  0.0000
13    F  0.0000  0.9801  0.9900  0.9703
14    F  0.9801  0.9900  1.0000  0.9801
15    G  0.0000  0.0000  0.0000  0.0000


In [8]:
path = [0, 4, 8, 9, 13, 14]                              # (1)
actions = [1, 1, 2, 1, 2, 2]
learned = Q[path, actions]                               # (2)
exact = gamma ** np.arange(5, -1, -1)                    # (3)
print("Q(s, a) on path =", learned)
print("gamma powers    =", exact)
assert np.allclose(learned, exact)

Q(s, a) on path = [0.9510 0.9606 0.9703 0.9801 0.9900 1.0000]
gamma powers    = [0.9510 0.9606 0.9703 0.9801 0.9900 1.0000]


## Шаг 7. Проверяем жадную политику

In [9]:
wins_eval = 0
for _ in range(n_eval):                                  # (1)
    s, _ = env.reset()
    done = False
    while not done:
        a = int(np.argmax(Q[s]))                         # (2)
        s, r, term, trunc, _ = env.step(a)
        done = term or trunc
    wins_eval += r                                       # (3)
print(f"success rate = {wins_eval / n_eval:.3f}")

success rate = 1.000


In [10]:
policy = np.argmax(Q, axis=1)                            # (1)
for i in range(4):
    row = []
    for j in range(4):
        s = 4 * i + j                                    # (2)
        terminal = cells[s] in "HG"
        symbol = cells[s] if terminal else ARROWS[policy[s]]
        row.append(symbol)                               # (3)
    print(" ".join(row))

↓ ← ← ←
↓ H ↓ H
→ ↓ ↓ H
H → → G
